# Desplegar Feactures Churn Client
Despliegue de caracteristicas y modelo para realizar predicciones en tiempo real via llamdas REST API.
Como las predicciones son realizadas en el front de aplicaciones cliente la respuesta debe ser de baja latencia y alta concurrencia.

Servir caracteristicas y modelo:
- Caracteristicas disponibles con baja latencia a traves de Databricks tablas online.
- Desplegar el modelo registrado en UC a un endpoint servicio de modelo de baja latencia.

## 1. Configuracion

In [0]:
catalog = "medallion_dev"
schema = "gold"
model_name = "customer_churn_classifier"
online_store_name = "churnonlinestore"

dbutils.widgets.text("model_full_name", f"{catalog}.{schema}.{model_name}", "Modelo nombre completo: catalog.schema.name") 
dbutils.widgets.text("model_version", "1", "Modelo Version (@Champion)")
dbutils.widgets.text("online_store_name", online_store_name, "Online Store Name")
dbutils.widgets.dropdown("drop_online_store", "False", ["True", "False"], "Reinciar Tabla(s) Online")
dbutils.widgets.dropdown("smoke_test", "False", ["True", "False"], "Bandera Smoke Test")

In [0]:
is_smoke_test = dbutils.widgets.get("smoke_test").lower() == "true"
online_store_name = dbutils.widgets.get("online_store_name") 

model_full_name = dbutils.widgets.get("model_full_name")
model_version = int(dbutils.widgets.get("model_version"))
model_endpoint = (f"{model_full_name}_endpoint")[:50]
fs_endpoint_churn = "churn_user_features_endpoint"

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"""USE `{catalog}`.`{schema}`""")

## 2. Servicio caracteristicas con Databricks Lakebase's tablas en linea
- Documentación:  ([AWS](https://docs.databricks.com/aws/en/machine-learning/feature-store/online-feature-store)|[Azure](https://learn.microsoft.com/en-us/azure/databricks/machine-learning/feature-store/online-feature-store)).
- Permisos: ([AWS](https://docs.databricks.com/en/machine-learning/feature-store/online-tables.html#user-permissions)|[Azure](https://learn.microsoft.com/azure/databricks/machine-learning/feature-store/online-tables#user-permissions)).
- Tablas de caracteristicas: [Databricks Feature Engineering SDK](https://api-docs.databricks.com/python/feature-engineering/latest/index.html).

In [0]:
%sql
ALTER TABLE churn_user_features SET TBLPROPERTIES (delta.enableChangeDataFeed = true);
ALTER TABLE churn_user_features ALTER COLUMN user_id SET NOT NULL;

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient
from time import time

# Initialize the client
fe = FeatureEngineeringClient()

print(f'Using online store: {online_store_name}')
print(f'Using fs endoint churn: {fs_endpoint_churn}')

# Check if exists
online_store = fe.get_online_store(name=online_store_name)

if online_store:
  print(f"Online store exists:")
  print(f"Store: {online_store.name}, State: {online_store.state}, Capacity: {online_store.capacity}")

  # Update the capacity of online store
  # updated_store = fe.update_online_store(name=online_store_name,
  #     capacity="CU_2"  # Upgrade to higher capacity
  # )

  if dbutils.widgets.get("drop_online_store") == "True" and not is_smoke_test:
    # Drop & wait
    fe.delete_online_store(name=online_store_name)
    time.sleep(60)

    print(f"Dropping/Recreating it.")
    online_store = fe.create_online_store(name=online_store_name,
      capacity="CU_1"
    )
elif not online_store and not is_smoke_test:
  print(f"Creating Online store: {online_store_name}")
  online_store = fe.create_online_store(name=online_store_name,
      capacity="CU_1"  # Valid options: "CU_1", "CU_2", "CU_4", "CU_8"
  )

In [0]:
if not is_smoke_test:
    print(f"Publishing feature table to online store...")
    max_retries = 5
    retry_count = 0
    while retry_count < max_retries:
        try:
            publish_state_user = fe.publish_table(online_store=online_store,
                source_table_name=f"{catalog}.{schema}.churn_user_features",
                online_table_name=f"{catalog}.{schema}.churn_user_features_online",
                # streaming=True
            )
            break
        except Exception as e:
            if "feature sync is currently in progress" in str(e):
                print("Sincronizar caracteristicas en progreso, reintentando...")
                retry_count += 1
                time.sleep(20 + (retry_count * 10))  # Wait x seconds before retrying
            else:
                raise e
    else:
        print("Fallo al publicar despues de intentar varias veces.")